# Fine Tuning with Optuna Framework

Esta versión del reconfigura los settings:
- Setting 1: Características del proyecto
- Setting 2: Características del proyecto + atributos e indicadores de la entidad
- Setting 3: Características del proyecto + atributos del contratista
- Setting 4: Características del proyecto + indicadores territoriales
- Setting 5: Características del proyecto + todas las características

In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
#!pip freeze > requirements.txt

In [3]:
!pip install optuna --quiet
!pip install optuna-integration[sklearn]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 4.0 MB/s eta 0:00:00


In [4]:
# Data Science Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


# Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import KFold


# Train model
# from ml_pipeline import *

# Utils
import warnings
import os
import json
import joblib


pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=FutureWarning)

ImportError: generic_type: type "ObjSense" is already registered!

In [ ]:
# Configurar conexion con google drive y con carpeta de google colab
from google.colab import drive
drive.mount('/content/drive')
workin_dir = '/content/drive/MyDrive/Colab Notebooks/Prediccion de Adiciones Proyectos de Infraestructura'
os.chdir(workin_dir)
import sys
# Add the directory containing the .py file to the Python path
sys.path.append(workin_dir) # Replace with the actual path if different

# Now you can import functions from your .py file
from ml_pipeline import *

In [ ]:
# Load list of columns
with open('Data/projectsColumns.txt', 'r') as f:
    projectsColumns = f.read().splitlines()
with open('Data/ownerColumns.txt', 'r') as f:
    ownerColumns = f.read().splitlines()
with open('Data/contractorColumns.txt', 'r') as f:
    contractorColumns = f.read().splitlines()
with open('Data/terridataColumns.txt', 'r') as f:
    terridataColumns = f.read().splitlines()
with open('Data/outcomeColumns.txt', 'r') as f:
    outcomeColumns = f.read().splitlines()

In [ ]:
df = pd.read_csv('Data/trainData.csv')   # Load the data
df = df[(df['contractValueMw(G1)'] >= 1e3) & (df['totalCumContracts(G2)'] >= 5)].copy().reset_index()  # Filter out rows with contractValueMw(G1) <= 1000
#df = df[df['totalCumContracts(G2)'] >= 10].copy().reset_index()  # Filter out rows with contractValueMw(G1) <= 1000
df.drop(columns=['index'], inplace=True)
df.info()
# convert contractMonthSigned(G2) and contractMonthStart(G2) to object
#df['contractMonthSigned(G2)'] = df['contractMonthSigned(G2)'].astype(str)
#df['contractMonthStart(G2)'] = df['contractMonthStart(G2)'].astype(str)

## Análisis Exploratorio de Datos

In [ ]:
df.columns[df.columns.str.contains('G1')]

### Todas las variables

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.select_dtypes(include=['int64', 'float64']).corr(method='spearman').round(2),
           cmap='coolwarm',
           annot=True,
           annot_kws={'size': 6},
           xticklabels=True,  # Mostrar labels
           yticklabels=True)

# Configurar tamaño después
plt.xticks(fontsize=7)  # Tamaño y rotación eje X
plt.yticks(fontsize=7)
plt.show()

Se plantea un torneo donde se enfrentan las variables que tienen un correlación superior a 0.7. Se enfrentan las variables altamente correlacionadas en un proceso de validación cruzada para mirar cuál tiene mejor desempeño.

In [ ]:
# Variables con datos faltantes
df.isnull().sum()

### Grupo 1

In [ ]:
projectsColumns

In [ ]:
# Vertical barplot
plt.figure(figsize=(10, 5))
sns.barplot(df['contractMethodType(G1)'])

In [ ]:
fig, ax = plt.subplots(figsize=(14, 16), ncols=3, nrows=4)
sns.histplot(df['contractValueMw(G1)'], kde=True, ax=ax[0,0])
sns.histplot(df['contractDurationDays(G1)'], kde=True, ax=ax[0,1])
sns.histplot(df['projectIntensityMw(G1)'], kde=True, ax=ax[0,2])
sns.histplot(df['prebidValueMw(G1)'], kde=True, ax=ax[1,0])
sns.histplot(df['bidContractRatio(G1)'], kde=True, ax=ax[1,1])
df.groupby('contractMonthSigned(G1)').size().plot(kind='bar', ax=ax[1,2])
df.groupby('contractMonthStart(G1)').size().plot(kind='bar', ax=ax[2,0])
df.groupby('electoralYear(G1)').size().plot(kind='bar', ax=ax[2,1])
df.groupby('preelectoralYear(G1)').size().plot(kind='bar', ax=ax[2,2])
df.groupby('workCategory(G1)').size().plot(kind='bar', ax=ax[3,0])
df.groupby('contractMethodType(G1)').size().plot(kind='bar', ax=ax[3,1])
df.groupby('databaseSource(G1)').size().plot(kind='bar', ax=ax[3,2])
plt.show()

In [ ]:
sns.heatmap(df[projectsColumns].select_dtypes(include=['int64', 'float64']).corr(method='spearman').round(2),
           cmap='coolwarm',
           annot=True,
           annot_kws={'size': 10},
           xticklabels=True,  # Mostrar labels
           yticklabels=True)

# Configurar tamaño después
plt.xticks(fontsize=10)  # Tamaño y rotación eje X
plt.yticks(fontsize=10)
plt.show()

### Grupo 2

In [ ]:
ownerColumns

In [ ]:
# Indicadores de concetración (2)
fig, ax = plt.subplots(figsize=(8, 4), ncols=2, nrows=1)
sns.histplot(df['histHHINumProject(G2)'], kde=True, ax=ax[0])
sns.histplot(df['histHHIValueProject(G2)'], kde=True, ax=ax[1])
plt.show()

In [ ]:
# Indicadores de desviaciones
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, nrows=1)
sns.histplot(df['histPctDesvTiempo(G2)'], kde=True, ax=ax[0])
sns.histplot(df['histPctDesvCost(G2)'], kde=True, ax=ax[1])
sns.histplot(df['histPctDesvTimeAndCost(G2)'], kde=True, ax=ax[2])
plt.show()

In [ ]:
# Indicadores de experiencia
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, nrows=1)
sns.histplot(df['histPctDirectContracts(G2)'], kde=True, ax=ax[0])
sns.histplot(df['histPctTenderContracts(G2)'], kde=True, ax=ax[1])
sns.histplot(df['totalCumContracts(G2)'], kde=True, ax=ax[2])
plt.show()

In [ ]:
# Indicadores de concetración (2)
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, nrows=1)
sns.histplot(df['histAvgContractValue(G2)'], kde=True, ax=ax[0])
sns.histplot(df['histAvgContractDuration(G2)'], kde=True, ax=ax[1])
sns.histplot(df['histAvgProjectIntensity(G2)'], kde=True, ax=ax[2])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4), ncols=3, nrows=1)
sns.histplot(df['histAvgBidRatio(G2)'], kde=True, ax=ax[0])
sns.histplot(df['histAvgTimeDeviation(G2)'], kde=True, ax=ax[1])
sns.histplot(df['histAvgCostDeviation(G2)'], kde=True, ax=ax[2])
plt.show()

In [ ]:
sns.heatmap(df[ownerColumns].select_dtypes(include=['int64', 'float64']).corr(method='spearman').round(2),
           cmap='coolwarm',
           annot=True,
           annot_kws={'size': 8},
           xticklabels=True,  # Mostrar labels
           yticklabels=True)
# Configurar tamaño después
plt.xticks(fontsize=10)  # Tamaño y rotación eje X
plt.yticks(fontsize=10)
plt.show()

### Grupo 4

In [ ]:
sns.heatmap(df[terridataColumns].select_dtypes(include=['int64', 'float64']).corr(method='spearman').round(2),
           cmap='coolwarm',
           annot=True,
           annot_kws={'size': 10},
           xticklabels=True,  # Mostrar labels
           yticklabels=True)

# Configurar tamaño después
plt.xticks(fontsize=10)  # Tamaño y rotación eje X
plt.yticks(fontsize=10)
plt.show()

## Recursive feature elimination

In [ ]:
# y = df['haveCostDeviation(G5)']
# X = df[vars_s5]

In [ ]:
# Feature selection with xgboost
# num_cols = X.select_dtypes(include=['int64', 'float64']).columns
# cat_cols = X.select_dtypes(include=['object']).columns
# preprocesor = create_preprocessing_pipeline(numeric_columns=num_cols, categorical_columns=cat_cols)
# pipe_rfe = Pipeline(
#     [
#        ('preprocesor', preprocesor),
#         ('rfe', RFE(estimator=XGBClassifier(random_state=42), n_features_to_select=20))
#     ]
# )
# pipe_rfe.fit(X, y)

In [ ]:
# ranking = pipe_rfe.named_steps["rfe"]
# preprocesor_rfe = pipe_rfe.named_steps["preprocesor"]
# ranking_var = pd.DataFrame(
#     {
#         "Variable": preprocesor_rfe.get_feature_names_out(),
#         "Ranking": ranking.ranking_
#     }
# )
# ranking_var.sort_values(by="Ranking", ascending=True, inplace=True)
# ranking_var.reset_index(drop=True, inplace=True)
# ranking_var.head(30)

## Selection by group

In [ ]:
def torneo_vars(model,
                df,
                out_come_var,
                torneo_vars,
                base_vars,
                grup_vars,
                scoring='roc_auc',
                random_state=42):

  n_jobs=-1
  results_torneo_var = {}
  stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)
  for t in torneo_vars.keys():
    print("="*10 + f" Torneo {t} " + "="*10)
    set_t_vars = base_vars + list(set(grup_vars) - set(torneo_vars[t]))
    for var in torneo_vars[t]:
      temp_vars = set_t_vars + [var]
      print(f"Resultado con la variable {var}")
      # Train Xgboost
      X = df[temp_vars]
      y = df[out_come_var]
      num_cols = X.select_dtypes(include=['int64', 'float64']).columns
      cat_cols = X.select_dtypes(include=['object']).columns
      preprocesor = create_preprocessing_pipeline(numeric_columns=num_cols, categorical_columns=cat_cols)
      pipe_model = Pipeline(
          [
              ('preprocesor', preprocesor),
              ('classifier', model)
          ]
      )
      scores = cross_val_score(
                  pipe_model,
                  X,
                  y,
                  scoring=scoring,
                  cv=stratified_cv,
                  n_jobs=n_jobs
              )
      results_torneo_var[var] = {
          "Torneo":t,
          "Variable":var,
          "scores":scores,
          "mean": scores.mean().round(4),
          "std": scores.std().round(4)
      }
      # print score
      print(f"Mean: {scores.mean():.4f} +- {scores.std():.4f}")

    if len(torneo_vars[t]) == 2:
      score_var1 = results_torneo_var[torneo_vars[t][0]]["scores"]
      score_var2 = results_torneo_var[torneo_vars[t][1]]["scores"]


      # Si las varianzas son diferentes (Welch's t-test)
      t_stat, p_value = stats.ttest_ind(score_var1, score_var2, equal_var=False)

      print(f"T-statistic: {t_stat:.4f}")
      print(f"P-value: {p_value:.4f}")

    # Hacer anova
    if len(torneo_vars[t]) > 2:
      f_stat, p_value = stats.f_oneway(*[results_torneo_var[var]["scores"] for var in torneo_vars[t]])
      print(f"F-statistic: {f_stat:.4f}")
      print(f"P-value: {p_value:.4f}")

  return results_torneo_var

### Grupo 1

In [ ]:
projectsColumns

In [ ]:
torneo_project_vars = {
    'T1':['prebidValueMw(G1)','contractValueMw(G1)','projectIntensityMw(G1)'],
    'T2':['contractMonthSigned(G1)','contractMonthStart(G1)']
}
model_torneo = XGBClassifier(random_state=42)
base_vars = []
grup_vars = projectsColumns
out_come_var = 'haveCostDeviation(G5)'
results_torneo_owner_var = torneo_vars(model_torneo,
                                       df,
                                       out_come_var,
                                       torneo_project_vars,
                                       base_vars,
                                       grup_vars)

In [ ]:
del_g1 = ['prebidValueMw(G1)','contractValueMw(G1)'#, # T1
          #'contractMonthStart(G1)'
          ] # T2

### Grupo 2

In [ ]:
model_torneo = XGBClassifier(random_state=42)
base_vars = list(set(projectsColumns) - set(del_g1))
grup_vars = ownerColumns
torneo_owner_vars = {
    'T1':['histHHINumProject(G2)','histHHIValueProject(G2)',],
    'T2':['histPctDesvTiempo(G2)','histPctDesvCost(G2)','histPctDesvTimeAndCost(G2)','histAvgCostDeviation(G2)','histAvgTimeDeviation(G2)'],
    'T3':['histAvgProjectIntensity(G2)','histAvgContractDuration(G2)','histAvgContractValue(G2)'],
    'T4':['penaltiesAccumulated(G2)','penaltiesRecent(G2)'],
    'T5':['ownerDepartment(G2)','ownerRegion(G2)']
}
out_come_var = 'haveCostDeviation(G5)'
results_torneo_owner_var = torneo_vars(model_torneo,
                                       df,
                                       out_come_var,
                                       torneo_owner_vars,
                                       base_vars,
                                       grup_vars)

In [ ]:
del_g2 = ['histHHINumProject(G2)', # T1
          'histPctDesvTimeAndCost(G2)','histAvgCostDeviation(G2)','histAvgTimeDeviation(G2)', # T2
          #'histAvgContractValue(G2)',
          'histAvgContractDuration(G2)', # T3
          'histAvgProjectIntensity(G2)',
          'penaltiesAccumulated(G2)', # T4
          'ownerRegion(G2)' # T5
          ]

### Grupo 3

In [ ]:
torneo_contractor_vars = {
    'T1':['contractorDepartment(G3)','contractorRegion(G3)']
}
model_torneo = XGBClassifier(random_state=42)
base_vars = list(set(projectsColumns) - set(del_g1))
grup_vars = contractorColumns
out_come_var = 'haveCostDeviation(G5)'
results_torneo_owner_var = torneo_vars(model_torneo,
                                       df,
                                       out_come_var,
                                       torneo_contractor_vars,
                                       base_vars,
                                       grup_vars)

In [ ]:
del_g3 = ['contractorRegion(G3)']

### Grupo 4

In [ ]:
terridataColumns

In [ ]:
torneo_terridata_vars = {
    'T1':['managementComponent(G4)','openGovernmentTransparency(G4)'],
    'T2':['gdp(G4)','gdpByEconomicActivityConstruction(G4)']
}
model_torneo = XGBClassifier(random_state=42)
base_vars = list(set(projectsColumns) - set(del_g1))
grup_vars = terridataColumns
out_come_var = 'haveCostDeviation(G5)'
results_torneo_owner_var = torneo_vars(model_torneo,
                                       df,
                                       out_come_var,
                                       torneo_terridata_vars,
                                       base_vars,
                                       grup_vars)

In [ ]:
del_g4 = ['gdp(G4)']

## Configuración de variables y escenarios

In [ ]:
projectsColumns = list(set(projectsColumns) - set(del_g1))
ownerColumns = list(set(ownerColumns) - set(del_g2))
contractorColumns = list(set(contractorColumns) - set(del_g3))
terridataColumns = list(set(terridataColumns) - set(del_g4))

In [ ]:
vars_s1 = projectsColumns
vars_s2 = projectsColumns + ownerColumns
vars_s3 = projectsColumns + contractorColumns
vars_s4 = projectsColumns + terridataColumns
vars_s5 = projectsColumns + ownerColumns + contractorColumns + terridataColumns
set_vars = [vars_s1, vars_s2, vars_s3, vars_s4, vars_s5]

## Cost Deviation

### Logistic Regression

In [ ]:
param_suggest_dict_lr = {
    'penalty': lambda trial: trial.suggest_categorical('penalty', ['l1', 'l2']),
    'C': lambda trial: trial.suggest_loguniform('C', 1e-4, 1e4),
}
lr_model = LogisticRegression(max_iter=10000, solver='liblinear', random_state=42)

In [ ]:
lr_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

  lr_train_tuning_evaluate = train_tuning_evaluate_model(
    X_train=X_train,
    y_train=y_train,
    model=lr_model,
    model_name='Logistic Regression',
    stage=stage,
    param_suggest_dict=param_suggest_dict_lr,
    X_test=X_test,
    y_test=y_test,
    n_trials=30,  # Reducido para el ejemplo
    cv=5,          # Reducido para el ejemplo
    scoring='roc_auc',
    direction='maximize',
    timeout=None,
    resampling_strategy='smoteenn',  # Ejemplo con SMOTE
    random_state=42,
    plot_tuning_history=False)

  lr_results[stage] = lr_train_tuning_evaluate

In [ ]:
# Guardar los resultados
joblib.dump(lr_results, 'Models V2/cost_deviation_lr_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
# Cargar el diccionario
lr_results = joblib.load('Models V2/cost_deviation_lr_results.joblib')
# print("Diccionario complejo cargado exitosamente")

### Random Forest

In [ ]:
param_suggest_dict_rf = {
    # Profundidad máxima: ampliando el rango para explorar más posibilidades
    'max_depth': lambda trial: trial.suggest_int('max_depth', 5, 50),

    # Número de estimadores: mantener el rango original pero con exploración continua
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 1000),

    # Samples para dividir: explorar valores con distribución logarítmica
    'min_samples_split': lambda trial: trial.suggest_int('min_samples_split', 2, 20),

    # Samples en hojas: explorar valores con distribución logarítmica
    'min_samples_leaf': lambda trial: trial.suggest_int('min_samples_leaf', 1, 10),

    # Características a considerar: explorar valores continuos y categóricos
    'max_features': lambda trial: trial.suggest_categorical('max_features', ['sqrt', 'log2'])
                    if trial.suggest_categorical('max_features_type', ['auto', 'float']) == 'auto'
                    else trial.suggest_float('max_features_float', 0.1, 0.9),

    # Parámetros adicionales que podrían mejorar el rendimiento
    'bootstrap': lambda trial: trial.suggest_categorical('bootstrap', [True, False]),
}

In [ ]:
rf_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  rf_model = RandomForestClassifier(random_state=42)
  # Usar la función para optimizar el modelo
  rf_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=rf_model,
      model_name='Random Forest',
      stage=stage,
      param_suggest_dict=param_suggest_dict_rf,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
  )
  rf_results[stage] = rf_train_tuning_evaluate

In [ ]:
# Guardar los resultados del random forest
joblib.dump(rf_results, 'Models V2/cost_deviation_rf_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
rf_results = joblib.load('Models V2/cost_deviation_rf_results.joblib')

### Extreme Gradient Boosting

In [ ]:
# Definir el diccionario de sugerencias de parámetros para Optuna
param_suggest_dict_xgb = {
    'max_depth': lambda trial: trial.suggest_int('max_depth', 3, 20),
    'learning_rate': lambda trial: trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 1000),
    'min_child_weight': lambda trial: trial.suggest_int('min_child_weight', 1, 10),
    'subsample': lambda trial: trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': lambda trial: trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': lambda trial: trial.suggest_float('gamma', 0, 10)
}

In [ ]:
# Usar la función para optimizar el modelo
xgb_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  # Definir el modelo base
  xgb_model = XGBClassifier(random_state=42)
  xgb_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=xgb_model,
      model_name='XGBoost',
      stage=stage,
      param_suggest_dict=param_suggest_dict_xgb,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
      )
  xgb_results[stage] = xgb_train_tuning_evaluate

In [ ]:
# Guardar los resultados del extreme gradient boosting
joblib.dump(xgb_results, 'Models V2/cost_deviation_xgb_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
xgb_results = joblib.load('Models V2/cost_deviation_xgb_results.joblib')

### K Nearest Neighbors

In [ ]:
param_suggest_dict_knn = {
    # Número de vecinos: explorar un rango más amplio y continuo
    'n_neighbors': lambda trial: trial.suggest_int('n_neighbors', 3, 40),

    # Función de pesos
    'weights': lambda trial: trial.suggest_categorical('weights', ['uniform', 'distance']),

    # Métrica de distancia con manejo condicional para el parámetro p
    'metric': lambda trial: trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),

    # Parámetro p solo relevante cuando metric='minkowski'
    #'p': lambda trial: trial.suggest_float('p', 1.1, 1.9) if trial.params.get('metric') == 'minkowski' else 2,

    # # Algoritmo para calcular vecinos
    # 'algorithm': lambda trial: trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
    #                 if trial.params.get('metric') in ['euclidean', 'manhattan'] else 'auto',

    # # Tamaño de hoja - importante para ball_tree y kd_tree
    # 'leaf_size': lambda trial: trial.suggest_int('leaf_size', 10, 100)
    #              if trial.params.get('algorithm') in ['ball_tree', 'kd_tree'] else 30
}

In [ ]:
knn_results = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveCostDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  knn_model = KNeighborsClassifier()

  # Usar la función para optimizar el modelo
  knn_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=knn_model,
      model_name='KNN',
      stage=stage,
      param_suggest_dict=param_suggest_dict_knn,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False)
  knn_results[stage] = knn_train_tuning_evaluate

In [ ]:
# Guardar los resultados del KNN
joblib.dump(knn_results, 'Models V2/cost_deviation_knn_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
knn_results = joblib.load('Models V2/cost_deviation_knn_results.joblib')

### Cost Deviation: results

In [ ]:
lr_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  lr_metrics[str(i)] = lr_results[f'S{i}']['metrics']
lr_metrics = pd.DataFrame(lr_metrics).T

rf_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  rf_metrics[str(i)] = rf_results[f'S{i}']['metrics']
rf_metrics = pd.DataFrame(rf_metrics).T

xgb_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_metrics[str(i)] = xgb_results[f'S{i}']['metrics']
xgb_metrics = pd.DataFrame(xgb_metrics).T

knn_metrics = {}
for i in np.arange(1, len(set_vars)+1):
  knn_metrics[str(i)] = knn_results[f'S{i}']['metrics']
knn_metrics = pd.DataFrame(knn_metrics).T

metrics_cost_deviation = pd.concat([lr_metrics, rf_metrics, xgb_metrics, knn_metrics], axis=0)
for i in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
  metrics_cost_deviation[i] = pd.to_numeric(metrics_cost_deviation[i]*100, errors='coerce')

map_setting = {
    'S1': 'Project',
    'S2': 'Project + Owner',
    'S3': 'Project + Contractor',
    'S4': 'Project + Terridata',
    'S5': 'All'
}
metrics_cost_deviation['setting'] = metrics_cost_deviation['stage'].map(map_setting)

metrics_cost_deviation = metrics_cost_deviation[['model', 'stage', 'setting', 'precision', 'recall', 'f1', 'accuracy', 'roc_auc']]
metrics_cost_deviation.to_excel('Data/metrics_cost_deviation.xlsx', index=False)
metrics_cost_deviation

In [ ]:
# Save best parameters for cost deviation models
lr_cost_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  lr_cost_best_params[f'S{i}'] = lr_results[f'S{i}']['best_params']
lr_cost_best_params = pd.DataFrame(lr_cost_best_params).T.reset_index(names='setting')
lr_cost_best_params.to_excel('Data/lr_cost_best_params.xlsx', index=False)

rf_cost_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  rf_cost_best_params[f'S{i}'] = rf_results[f'S{i}']['best_params']
rf_cost_best_params = pd.DataFrame(rf_cost_best_params).T.reset_index(names='setting')
rf_cost_best_params.to_excel('Data/rf_cost_best_params.xlsx', index=False)

xgb_cost_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_cost_best_params[f'S{i}'] = xgb_results[f'S{i}']['best_params']
xgb_cost_best_params = pd.DataFrame(xgb_cost_best_params).T.reset_index(names='setting')
xgb_cost_best_params.to_excel('Data/xgb_cost_best_params.xlsx', index=False)

knn_cost_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  knn_cost_best_params[f'S{i}'] = knn_results[f'S{i}']['best_params']
knn_cost_best_params = pd.DataFrame(knn_cost_best_params).T.reset_index(names='setting')
knn_cost_best_params.to_excel('Data/knn_cost_best_params.xlsx', index=False)

## Time Deviation

### Logistic Regression

In [ ]:
param_suggest_dict_lr = {
    'penalty': lambda trial: trial.suggest_categorical('penalty', ['l1', 'l2']),
    'C': lambda trial: trial.suggest_loguniform('C', 1e-4, 1e4),
}
lr_model = LogisticRegression(max_iter=10000, solver='liblinear', random_state=42)

lr_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

  lr_train_tuning_evaluate = train_tuning_evaluate_model(
    X_train=X_train,
    y_train=y_train,
    model=lr_model,
    model_name='Logistic Regression',
    stage=stage,
    param_suggest_dict=param_suggest_dict_lr,
    X_test=X_test,
    y_test=y_test,
    n_trials=30,  # Reducido para el ejemplo
    cv=5,          # Reducido para el ejemplo
    scoring='roc_auc',
    direction='maximize',
    timeout=None,
    resampling_strategy='smoteenn',  # Ejemplo con SMOTE
    random_state=42,
    plot_tuning_history=False)

  lr_results_time_deviation[stage] = lr_train_tuning_evaluate

In [ ]:
# Guardar los resultados
joblib.dump(lr_results_time_deviation, 'Models V2/time_deviation_lr_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
lr_results_time_deviation = joblib.load('Models V2/time_deviation_lr_results.joblib')

In [ ]:
# Cargar el diccionario
# loaded_dict = joblib.load('Models/cost_deviation_lr_results.joblib')
# print("Diccionario complejo cargado exitosamente")

### Random Forest

In [ ]:
param_suggest_dict_rf = {
    # Profundidad máxima: ampliando el rango para explorar más posibilidades
    'max_depth': lambda trial: trial.suggest_int('max_depth', 5, 50),

    # Número de estimadores: mantener el rango original pero con exploración continua
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 1000),

    # Samples para dividir: explorar valores con distribución logarítmica
    'min_samples_split': lambda trial: trial.suggest_int('min_samples_split', 2, 20),

    # Samples en hojas: explorar valores con distribución logarítmica
    'min_samples_leaf': lambda trial: trial.suggest_int('min_samples_leaf', 1, 10),

    # Características a considerar: explorar valores continuos y categóricos
    'max_features': lambda trial: trial.suggest_categorical('max_features', ['sqrt', 'log2'])
                    if trial.suggest_categorical('max_features_type', ['auto', 'float']) == 'auto'
                    else trial.suggest_float('max_features_float', 0.1, 0.9),

    # Parámetros adicionales que podrían mejorar el rendimiento
    'bootstrap': lambda trial: trial.suggest_categorical('bootstrap', [True, False]),
}



rf_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']

  # Split data into training and testing sets
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  rf_model = RandomForestClassifier(random_state=42)
  # Usar la función para optimizar el modelo
  rf_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=rf_model,
      model_name='Random Forest',
      stage=stage,
      param_suggest_dict=param_suggest_dict_rf,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
  )
  rf_results_time_deviation[stage] = rf_train_tuning_evaluate

In [ ]:
# Guardar los resultados del random forest
joblib.dump(rf_results_time_deviation, 'Models V2/time_deviation_rf_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
rf_results_time_deviation = joblib.load('Models V2/time_deviation_rf_results.joblib')

### Extreme Gradient Boosting

In [ ]:
# Definir el diccionario de sugerencias de parámetros para Optuna
param_suggest_dict_xgb = {
    'max_depth': lambda trial: trial.suggest_int('max_depth', 3, 20),
    'learning_rate': lambda trial: trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
    'n_estimators': lambda trial: trial.suggest_int('n_estimators', 50, 1000),
    'min_child_weight': lambda trial: trial.suggest_int('min_child_weight', 1, 10),
    'subsample': lambda trial: trial.suggest_float('subsample', 0.5, 1.0),
    'colsample_bytree': lambda trial: trial.suggest_float('colsample_bytree', 0.5, 1.0),
    'gamma': lambda trial: trial.suggest_float('gamma', 0, 10)
}

# Usar la función para optimizar el modelo
xgb_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  # Definir el modelo base
  xgb_model = XGBClassifier(random_state=42)
  xgb_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=xgb_model,
      model_name='XGBoost',
      stage=stage,
      param_suggest_dict=param_suggest_dict_xgb,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False
      )
  xgb_results_time_deviation[stage] = xgb_train_tuning_evaluate

In [ ]:
# Guardar los resultados del extreme gradient boosting
joblib.dump(xgb_results_time_deviation, 'Models V2/time_deviation_xgb_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
xgb_results_time_deviation = joblib.load('Models V2/time_deviation_xgb_results.joblib')

### K Nearest Neighbors

In [ ]:
param_suggest_dict_knn = {
    # Número de vecinos: explorar un rango más amplio y continuo
    'n_neighbors': lambda trial: trial.suggest_int('n_neighbors', 3, 40),

    # Función de pesos
    'weights': lambda trial: trial.suggest_categorical('weights', ['uniform', 'distance']),

    # Métrica de distancia con manejo condicional para el parámetro p
    'metric': lambda trial: trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),

    # Parámetro p solo relevante cuando metric='minkowski'
    #'p': lambda trial: trial.suggest_float('p', 1.1, 1.9) if trial.params.get('metric') == 'minkowski' else 2,

    # # Algoritmo para calcular vecinos
    # 'algorithm': lambda trial: trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute'])
    #                 if trial.params.get('metric') in ['euclidean', 'manhattan'] else 'auto',

    # # Tamaño de hoja - importante para ball_tree y kd_tree
    # 'leaf_size': lambda trial: trial.suggest_int('leaf_size', 10, 100)
    #              if trial.params.get('algorithm') in ['ball_tree', 'kd_tree'] else 30
}
knn_results_time_deviation = {}
for i in range(len(set_vars)):
  stage = 'S' + str(i+1)
  X = df[set_vars[i]]
  y = df['haveTimeDeviation(G5)']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
  knn_model = KNeighborsClassifier()

  # Usar la función para optimizar el modelo
  knn_train_tuning_evaluate = train_tuning_evaluate_model(
      X_train=X_train,
      y_train=y_train,
      model=knn_model,
      model_name='KNN',
      stage=stage,
      param_suggest_dict=param_suggest_dict_knn,
      X_test=X_test,
      y_test=y_test,
      n_trials=30,  # Reducido para el ejemplo
      cv=5,          # Reducido para el ejemplo
      scoring='roc_auc',
      direction='maximize',
      timeout=None,
      resampling_strategy='smoteenn',  # Ejemplo con SMOTE
      random_state=42,
      plot_tuning_history=False)
  knn_results_time_deviation[stage] = knn_train_tuning_evaluate

In [ ]:
# Guardar los resultados del KNN
joblib.dump(knn_results_time_deviation, 'Models V2/time_deviation_knn_results.joblib')
print("Diccionario complejo guardado exitosamente")

In [ ]:
knn_results_time_deviation = joblib.load('Models V2/time_deviation_knn_results.joblib')

### Time Deviation: results

In [ ]:
lr_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  lr_metrics_time_deviation[str(i)] = lr_results_time_deviation[f'S{i}']['metrics']
lr_metrics_time_deviation = pd.DataFrame(lr_metrics_time_deviation).T

rf_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  rf_metrics_time_deviation[str(i)] = rf_results_time_deviation[f'S{i}']['metrics']
rf_metrics_time_deviation = pd.DataFrame(rf_metrics_time_deviation).T

xgb_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_metrics_time_deviation[str(i)] = xgb_results_time_deviation[f'S{i}']['metrics']
xgb_metrics_time_deviation = pd.DataFrame(xgb_metrics_time_deviation).T

knn_metrics_time_deviation = {}
for i in np.arange(1, len(set_vars)+1):
  knn_metrics_time_deviation[str(i)] = knn_results_time_deviation[f'S{i}']['metrics']
knn_metrics_time_deviation = pd.DataFrame(knn_metrics_time_deviation).T

metrics_time_deviation = pd.concat([lr_metrics_time_deviation,
                                     rf_metrics_time_deviation,
                                     xgb_metrics_time_deviation,
                                     knn_metrics_time_deviation], axis=0)
for i in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
  metrics_time_deviation[i] = pd.to_numeric(metrics_time_deviation[i]*100, errors='coerce')

map_setting = {
    'S1': 'Project',
    'S2': 'Project + Owner',
    'S3': 'Project + Contractor',
    'S4': 'Project + Terridata',
    'S5': 'All'
}
metrics_time_deviation['setting'] = metrics_time_deviation['stage'].map(map_setting)

metrics_time_deviation = metrics_time_deviation[['model', 'stage', 'setting', 'precision', 'recall', 'f1', 'accuracy', 'roc_auc']]
metrics_time_deviation.to_excel('Data/metrics_time_deviation.xlsx', index=False)
metrics_time_deviation

In [ ]:
# Save best parameters for cost deviation models
lr_time_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  lr_time_best_params[f'S{i}'] = lr_results_time_deviation[f'S{i}']['best_params']
lr_time_best_params = pd.DataFrame(lr_time_best_params).T.reset_index(names='setting')
lr_time_best_params.to_excel('Data/lr_time_best_params.xlsx', index=False)

rf_time_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  rf_time_best_params[f'S{i}'] = rf_results_time_deviation[f'S{i}']['best_params']
rf_time_best_params = pd.DataFrame(rf_time_best_params).T.reset_index(names='setting')
rf_time_best_params.to_excel('Data/rf_time_best_params.xlsx', index=False)

xgb_time_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  xgb_time_best_params[f'S{i}'] = xgb_results_time_deviation[f'S{i}']['best_params']
xgb_time_best_params = pd.DataFrame(xgb_time_best_params).T.reset_index(names='setting')
xgb_time_best_params.to_excel('Data/xgb_time_best_params.xlsx', index=False)

knn_time_best_params = {}
for i in np.arange(1, len(set_vars)+1):
  knn_time_best_params[f'S{i}'] = knn_results_time_deviation[f'S{i}']['best_params']
knn_time_best_params = pd.DataFrame(knn_time_best_params).T.reset_index(names='setting')
knn_time_best_params.to_excel('Data/knn_time_best_params.xlsx', index=False)